In [16]:
# --- run me first ---
from pathlib import Path
import os
import sys

# project/notebooks -> project
if Path.cwd().name == "notebooks":
    os.chdir("..")

ROOT = Path.cwd()

# So `from src....` imports work
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("working from:", ROOT.name)

working from: project


In [17]:
# !pip install kaggle
# !pip install kagglehub

# Data Acquisition and Ingection

### Load and Prepare Transaction Data

This cell downloads the **PaySim synthetic financial transaction dataset** from Kaggle and prepares it for transaction network analysis.

- `nameOrig` is renamed to `sender`.

- `nameDest` is renamed to `receiver`.

- Relevant transaction fields such as amount, type, and fraud status are retained.

- A unique `transaction_id` is added to each transaction.

Each row therefore represents a directed transaction from a **sender → receiver**.

In [21]:
import kagglehub
import pandas as pd
from pathlib import Path

# Download PaySim
path = kagglehub.dataset_download("ealaxi/paysim1")

# Find and load CSV
csv_path = list(Path(path).glob("*.csv"))[0]
df = pd.read_csv(csv_path)

# Keep transaction-network columns
transactions = df[
    ["step", "nameOrig", "nameDest", "amount", "type", "isFraud"]
].copy()

# Rename columns
transactions = transactions.rename(columns={
    "nameOrig": "sender",
    "nameDest": "receiver",
    "isFraud": "is_fraud"
})

# Add transaction ID
transactions.insert(
    0,
    "transaction_id",
    range(1, len(transactions) + 1)
)

transactions.head(10)


,transaction_id,step,sender,receiver,amount,type,is_fraud
0,1,1,C1231006815,M1979787155,9839.64,PAYMENT,0
1,2,1,C1666544295,M2044282225,1864.28,PAYMENT,0
2,3,1,C1305486145,C553264065,181.00,TRANSFER,1
3,4,1,C840083671,C38997010,181.00,CASH_OUT,1
4,5,1,C2048537720,M1230701703,11668.14,PAYMENT,0
5,6,1,C90045638,M573487274,7817.71,PAYMENT,0
6,7,1,C154988899,M408069119,7107.77,PAYMENT,0
7,8,1,C1912850431,M633326333,7861.64,PAYMENT,0
8,9,1,C1265012928,M1176932104,4024.36,PAYMENT,0
9,10,1,C712410124,C195600860,5337.77,DEBIT,0


# Data Storage

In [23]:
import pandas as pd
from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR

print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)

# --------------------------------------------------
# 1. Define storage paths
# --------------------------------------------------
raw_csv_path = RAW_DATA_DIR / "transactions.csv"
processed_parquet_path = PROCESSED_DATA_DIR / "transactions.parquet"

# --------------------------------------------------
# 2. Save data
# --------------------------------------------------

# Raw data -> CSV
transactions.to_csv(raw_csv_path, index=False)
print("\nSaved raw CSV:")
print(raw_csv_path)

# Processed data -> Parquet
try:
    transactions.to_parquet(processed_parquet_path, index=False)
    print("\nSaved processed Parquet:")
    print(processed_parquet_path)
except Exception as e:
    print("\nParquet save failed:", e)

# --------------------------------------------------
# 3. Reload data
# --------------------------------------------------

transactions_csv = pd.read_csv(raw_csv_path)

print("\nReloaded CSV shape:", transactions_csv.shape)

if processed_parquet_path.exists():
    transactions_parquet = pd.read_parquet(processed_parquet_path)
    print("Reloaded Parquet shape:", transactions_parquet.shape)

# --------------------------------------------------
# 4. Validate reload
# --------------------------------------------------

expected_columns = [
    "transaction_id",
    "step",
    "sender",
    "receiver",
    "amount",
    "type",
    "is_fraud"
]

csv_checks = {
    "same_shape": transactions.shape == transactions_csv.shape,
    "columns_present": all(
        col in transactions_csv.columns
        for col in expected_columns
    ),
    "amount_is_numeric": pd.api.types.is_numeric_dtype(
        transactions_csv["amount"]
    )
}

print("\nCSV validation:")
print(csv_checks)

if processed_parquet_path.exists():

    parquet_checks = {
        "same_shape": transactions.shape == transactions_parquet.shape,
        "columns_present": all(
            col in transactions_parquet.columns
            for col in expected_columns
        ),
        "amount_is_numeric": pd.api.types.is_numeric_dtype(
            transactions_parquet["amount"]
        )
    }

    print("\nParquet validation:")
    print(parquet_checks)

# --------------------------------------------------
# 5. Preview stored data
# --------------------------------------------------

transactions_csv.head()

RAW_DATA_DIR: /Users/moctran/Desktop/NYU/BootcampIV/bootcamp_thuychau_tran/project/data/raw
PROCESSED_DATA_DIR: /Users/moctran/Desktop/NYU/BootcampIV/bootcamp_thuychau_tran/project/data/processed

Saved raw CSV:
/Users/moctran/Desktop/NYU/BootcampIV/bootcamp_thuychau_tran/project/data/raw/transactions.csv

Saved processed Parquet:
/Users/moctran/Desktop/NYU/BootcampIV/bootcamp_thuychau_tran/project/data/processed/transactions.parquet

Reloaded CSV shape: (6362620, 7)
Reloaded Parquet shape: (6362620, 7)

CSV validation:
{'same_shape': True, 'columns_present': True, 'amount_is_numeric': True}

Parquet validation:
{'same_shape': True, 'columns_present': True, 'amount_is_numeric': True}


,transaction_id,step,sender,receiver,amount,type,is_fraud
0,1,1,C1231006815,M1979787155,9839.64,PAYMENT,0
1,2,1,C1666544295,M2044282225,1864.28,PAYMENT,0
2,3,1,C1305486145,C553264065,181.00,TRANSFER,1
3,4,1,C840083671,C38997010,181.00,CASH_OUT,1
4,5,1,C2048537720,M1230701703,11668.14,PAYMENT,0
